# Is plan noise the cheapest route to diverse code? (analysis demo)

**Question.** Qwen3-8B (FP8 W8A8, thinking off) usually writes *one* default algorithm for a coding task. It uses a full sort
for the k-th smallest value, a hash map for two-sum, BFS for a grid path and Dijkstra for a weighted shortest path.
**Which knob buys CORRECT algorithmic diversity most cheaply?**

**The candidate: fresh per-sample plan noise (PN).** For every sample, one isotropic random unit direction `u` (a fresh
seed per sample) is added to the hidden states at the task-specification positions of decoder blocks 12-23. The edit
happens during the prompt prefill only. It is displacement-targeted: `h <- h + (alpha * n[l,p] - (h - h_twin) . u) u`,
so the state moves exactly `alpha` "ban units" along `u`. The ban unit `n[l,p]` is a per-block, per-position norm
carried over from the iteration-3 ban experiments. The settings are alpha = 0.5, 1, 2 and 4, plus 8 as a descriptive
extra.

**What PN is priced against:**
- temperature 1.3 and 1.6, and top-k 10;
- the nonce ban ("Do not use the Zorblax method ...");
- the novelty instruction ("Write a novel algorithm ...");
- a semantic seed (path tasks only);
- the reused iteration-3 signature vetoes: one-shot, persistent and iterated.

Every arm shares exact **common random numbers** (CRN): the same uniform stream per (task, template, slot). Two arms
therefore differ only by their knob.

**Statistic and price.**
- **Statistic:** correct-only Hill-1, rarefied at depth 25. Hill-1 is exp(Shannon entropy) of the mechanism labels of
  the *passing* programs, i.e. the effective number of distinct correct algorithms.
- **Randomisation hull:** for each knob class, mixtures of the reference arm and the knob's settings.
- **H5:** the largest Hill reachable on the hull at a pass cost of at most 5 points.
- **P15 / P20:** the cheapest pass cost at which Hill reaches 1.5 / 2.
- **Uncertainty:** 2,000-replicate bootstraps, with the best setting chosen inside every replicate.

**Pre-registered rule C1** on the six decisive tasks (E6 = T03, T05, T06, T10, T1, T2). C1 is CONFIRMED only if all
three hold:
- PN reaches H5 >= 1.5 on at least 3 tasks (*count1*);
- PN beats the best non-activation (NA) knob on at least 4 tasks (*count2*);
- the pooled margin D_bar has a 95% CI above 0.

**Full-run result: PARTIAL** (count1 2/6, count2 4/6, D_bar -0.22, 95% CI [-0.89, 0.30]). PN wins clearly on T03 and
T10. The reused iterated veto beats it on T06, and the semantic seed beats it on T2.

### What this notebook runs
`method.py` orchestrates eleven stages: prompts, inputs, screen, **gpu**, label, label_ec, commit, audit, **analyze**,
validity and report. The GPU stage generated 10,100 programs in 2.5 GPU-hours on a 16 GB FP8 GPU. That stage, the
labeller and the OpenRouter audits cannot run in Colab, so their saved output (the labelled programs) is this notebook's
input.

The notebook re-runs the **`analyze` stage** with the original code (`src/analysis/stats_step0.py`, `hull.py`,
`core.py`, `run_all.py`, `descriptive.py`, plus the figure code of `src/figures.py`) on the six decisive tasks. That is
99 (task, arm) cells and 5,020 labelled programs, including 1,620 reused iteration-3 veto rows.

* **Point estimates** (knob rows; H5 / P15 / P20 of every class) come out **identical** to the full run, and the
  notebook checks this.
* **Bootstrap replicate counts** are scaled down to fit a 10-minute budget, so CIs and bootstrap probabilities are
  noisier than in the paper. The full-run values are listed in the config cell.

In [ ]:
import subprocess, sys
def _pip(*a): subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *a])

# loguru -- NOT on Colab, always install (the original analysis code logs with it)
_pip('loguru==0.7.3')

# numpy, scipy, pandas, matplotlib -- pre-installed on Colab, install locally only (at Colab's versions)
if 'google.colab' not in sys.modules:
    _pip('numpy==2.0.2', 'scipy==1.16.3', 'pandas==2.2.2', 'matplotlib==3.10.0')

In [ ]:
from __future__ import annotations

# --- method.py's import block, as-is (the orchestrator; its subprocess stages are replaced by the cells below) ---
import argparse
import json
import subprocess
import sys
from pathlib import Path

from loguru import logger

# --- imports of the modules the `analyze` stage runs (src/cfg.py, src/analysis/*.py) ---
import itertools
import math
import time
import zlib
from collections import Counter, defaultdict
from datetime import datetime, timezone
from typing import Any, Sequence

import numpy as np

# --- notebook additions: tables and figures ---
import matplotlib.pyplot as plt
import pandas as pd

# method.py's logger set-up (stdout sink only; the original also wrote logs/method.log)
logger.remove()
logger.add(sys.stdout, level="INFO", format="{time:HH:mm:ss}|{level:<7}|{message}")
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 30)

In [ ]:
GITHUB_DATA_URL = "https://raw.githubusercontent.com/ai-inventor-papers/ai-invention-295f8e-prompt-bans-substitute-not-just-subtract/fork/run_0oWBKrKEGsmU/round-4/experiment-19/demo/mini_demo_data.json"
import http.client
import json
import urllib.request
from pathlib import Path

def load_data():
    try:
        with urllib.request.urlopen(GITHUB_DATA_URL, timeout=15) as response:
            return json.loads(response.read().decode())
    # Network errors, timeouts, a placeholder URL, bad JSON, a cut-off response.
    except (OSError, ValueError, http.client.HTTPException) as e:
        print("GitHub data unavailable, using local mini_demo_data.json:", e)
    local = Path("mini_demo_data.json")
    if local.exists(): return json.loads(local.read_text())
    raise FileNotFoundError("Could not load mini_demo_data.json")

In [ ]:
data = load_data()
meta = data["metadata"]
cells = data["datasets"][0]["examples"]   # one example per (task, arm) cell, with every labelled program of the cell
print(meta["title"])
print(f"{len(cells)} (task, arm) cells, {sum(len(c['rows']) for c in cells)} labelled programs")
print("model:", meta["model"])
print("one program row:", cells[0]["rows"][0])

## Configuration

The whole analysis is CPU-only. Two kinds of parameters set its cost:

* **Bootstrap sizes.** The full run used 2,000 primary replicates per task and 1,000 for each sensitivity bootstrap,
  run on 6 worker processes. Each replicate re-optimises every knob class's hull, at about 0.25 s per replicate for
  all six tasks on a 2-vCPU machine. The values below fit the 10-minute budget, and the full-run values are in the
  comments.
* **Monte-Carlo size of the population-rarefied Hill.** `N_SUB_POINT = 1000` (the full-run value) makes every point
  estimate bit-identical to the paper.

To reproduce the paper's bootstrap numbers, set every parameter to its full-run value. That takes about 30 minutes
here, because this notebook runs the bootstrap serially; the original spread it over 6 processes.

In [ ]:
# ---- bootstrap sizes (full-run values in the comments) ----
N_BOOT = 400           # primary bootstrap: replicates per task                       (full run: 2000)
N_BOOT_SENS = 50       # replicates of each sensitivity bootstrap s1, s3, s4, s5, s6, s8 (full run: 1000)
N_BOOT_C1 = 2000       # task-level resamples of the pooled margin D_bar in c1_decide  (full run: 2000)
KNOB_ROW_N_BOOT = 2000 # replicates of each knob row's Step-0 90% interval             (full run: 2000)
# ---- Monte-Carlo size of the population-rarefied Hill on the hull ----
N_SUB_POINT = 1000     # fixed uniform rows for the point estimates   (full run: 1000 -> identical to the paper)
N_SUB_BOOT = 200       # fixed uniform rows inside bootstrap replicates (full run: 200)
# ---- execution ----
WORKERS = 6            # replicates are split into seeded jobs per worker as in the full run; the jobs run serially here
RUN_HULL_UNIT_TESTS = True   # hull.py unit tests (a)-(d) + the full-simplex SLSQP diagnostic

## Constants, tasks and arms (`src/cfg.py`)

**Tasks and cells.** Each arm of a task is one **cell** of N = 50 programs: 5 prompt templates (W1-W5) x 10 CRN slots.

**Reference arms.**
- **TWIN**, the *neutral twin*, is the prompt frame of a ban with neutral nouns, e.g. "Do not use man for this task."
  It carries the prohibition wording without banning any algorithm, and it is the reference arm of this artifact's
  knobs.
- The reused veto rows were generated in iteration 3 by another engine (vLLM bf16 or HF). They are therefore priced
  against their own twin, **REUSED_TWIN**.

The table lists each arm's decoding and prompt settings.

In [ ]:
# --- from src/cfg.py ---
E6_TASKS = ("T03", "T05", "T06", "T10", "T1", "T2")   # frozen decisive event set
DEFAULT = meta["default_label"]                       # each task's default mechanism (cfg.DEFAULT, E6 subset)
ARMS = meta["arms"]                                   # arm -> decoding / prompt definition (cfg.ARMS, arms in this demo)
BOOT_SEED = 20261021


def utc_now() -> str:
    return datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%S.%fZ")


for t in E6_TASKS:
    print(f"{t:4s} {meta['task_names'][t]:52s} default: {DEFAULT[t]}")
print()
print("TWIN prompt of T03 (template W1):\n" + meta["prompts"]["T03"]["twin_W1_user_text"])
pd.DataFrame(ARMS).T

## Step-0 statistic: rarefied Hill-1 (`src/analysis/stats_step0.py`)

Hill-1 of a composition is `exp(Shannon entropy)`, i.e. the effective number of distinct ideas. Its plug-in value grows
with the sample size, so programs are compared at a fixed **depth of 25**. For one arm (one hull vertex), the code takes
the mean Hill-1 over 1,000 subsamples of 25 correct programs drawn *without* replacement (hypergeometric).
`boot_rarefied_hill1` gives the 90% interval of a knob row by resampling the programs. These three functions are copied
verbatim (the other helpers of the module are unused here).

In [ ]:
def _hill_rows(M: np.ndarray) -> np.ndarray:
    n = M.sum(axis=1, keepdims=True)
    p = np.where(M > 0, M / np.maximum(n, 1), 1.0)
    return np.exp(-(np.where(M > 0, (M / np.maximum(n, 1)) * np.log(p), 0.0)).sum(axis=1))


def rarefied_hill1(counts: Sequence[float], depth: int, n_sub: int = 1000,
                   rng: np.random.Generator | None = None) -> float:
    """Mean over n_sub subsamples of size `depth` drawn WITHOUT replacement of exp(Shannon) of the subsample."""
    c = np.asarray(counts, dtype=np.int64)
    c = c[c > 0]
    if c.sum() < depth or depth <= 0:
        return math.nan
    if len(c) == 1:
        return 1.0
    rng = rng or np.random.default_rng(0)
    M = rng.multivariate_hypergeometric(c, depth, size=n_sub)
    return float(_hill_rows(M.astype(float)).mean())


def boot_rarefied_hill1(counts: Sequence[float], depth: int, n_boot: int = 2000, n_sub: int = 200,
                        rng: np.random.Generator | None = None, conf: float = 0.90) -> tuple[float, float]:
    """Percentile interval of rarefied Hill-1 over multinomial resamples of the row (programs resampled)."""
    c = np.asarray(counts, dtype=np.int64)
    c = c[c > 0]
    n = int(c.sum())
    if n < depth:
        return (math.nan, math.nan)
    rng = rng or np.random.default_rng(1)
    p = c / n
    vals = np.empty(n_boot)
    for b in range(n_boot):
        cb = rng.multinomial(n, p)
        cb = cb[cb > 0]
        if len(cb) == 1:
            vals[b] = 1.0
            continue
        M = rng.multivariate_hypergeometric(cb, depth, size=n_sub)
        vals[b] = _hill_rows(M.astype(float)).mean()
    a = (1 - conf) / 2
    return (float(np.quantile(vals, a)), float(np.quantile(vals, 1 - a)))

## The randomisation hull (`src/analysis/hull.py`, verbatim)

A knob's price should not depend on which few settings were sampled. Instead of comparing single arms, each knob class
is priced on its **randomisation hull**: every mixture `lambda` of the reference arm and the knob's settings, i.e. "run
setting j on a random fraction lambda_j of the prompts".

**Value of a mixture.**
- correct-idea rates `r(lambda) = sum lambda_j r_j`;
- pass rate `pass(lambda) = sum lambda_j pass_j`;
- pass cost `100 (pass_0 - pass(lambda))`, in points.

**Hill on the hull.** Hill is the **population-rarefied** Hill-1 at depth 25: the mean of `exp(Shannon)` over multinomial
draws of 25 programs from the mixture's composition. All draws use one fixed matrix of uniforms (common random numbers),
so the objective is smooth in `lambda`.

**Search.**
1. Evaluate every pairwise edge of the simplex on a 0.025 grid.
2. Add the exact cost = 5 boundary point of each edge.
3. Refine the best edge by golden-section search.

From this search, **H5** is the best Hill at cost <= 5. **P15 / P20** is the smallest cost at which Hill reaches
1.5 / 2, or `None` if it never does.

In [ ]:
GRID = np.round(np.arange(0.0, 1.0 + 1e-9, 0.025), 6)


def fixed_uniforms(n_sub: int, depth: int = 25, seed: int = 20261021) -> np.ndarray:
    return np.random.default_rng(seed).random((n_sub, depth))


def pop_hill_loop(Q: np.ndarray, U: np.ndarray) -> np.ndarray:
    """Reference implementation of pop_hill (row loop); kept for the equivalence unit test."""
    G, C = Q.shape
    if C == 1:
        return np.ones(G)
    cdf = np.cumsum(Q, axis=1)
    cdf[:, -1] = 1.0 + 1e-12
    n_sub, depth = U.shape
    idx = np.empty((G, n_sub, depth), dtype=np.int16)
    flatU = U.ravel()
    for g in range(G):
        idx[g] = np.searchsorted(cdf[g], flatU, side="right").reshape(n_sub, depth)
    np.minimum(idx, C - 1, out=idx)
    H = np.zeros((G, n_sub))
    for c in range(C):
        cnt = (idx == c).sum(axis=2)
        pc = cnt / depth
        with np.errstate(divide="ignore", invalid="ignore"):
            H -= np.where(cnt > 0, pc * np.log(np.where(cnt > 0, pc, 1.0)), 0.0)
    return np.exp(H).mean(axis=1)


def pop_hill(Q: np.ndarray, U: np.ndarray) -> np.ndarray:
    """Q [G, C] compositions (rows sum to 1); U [n_sub, depth] -> [G] mean exp(Shannon) of multinomial(depth, q)
    drawn by inverse CDF from the fixed uniforms (one searchsorted over row-offset CDFs, one bincount)."""
    G, C = Q.shape
    if C == 1 or G == 0:
        return np.ones(G)
    cdf = np.cumsum(Q, axis=1)
    cdf[:, -1] = 1.0 + 1e-12
    n_sub, depth = U.shape
    off = 2.0 * np.arange(G, dtype=float)
    flat_cdf = (cdf + off[:, None]).ravel()
    u = (U.ravel()[None, :] + off[:, None]).ravel()
    pos = np.searchsorted(flat_cdf, u, side="right")
    idx = pos - np.repeat(np.arange(G) * C, n_sub * depth)
    np.minimum(idx, C - 1, out=idx)
    key = np.repeat(np.arange(G * n_sub), depth) * C + idx
    cnt = np.bincount(key, minlength=G * n_sub * C).reshape(G, n_sub, C)
    pc = cnt / depth
    with np.errstate(divide="ignore", invalid="ignore"):
        H = -np.where(cnt > 0, pc * np.log(np.where(cnt > 0, pc, 1.0)), 0.0).sum(axis=2)
    return np.exp(H).mean(axis=1)


class Hull:
    """Vertices: R [m, C] (correct-idea rates), pas [m] (pass rates); vertex 0 is the reference."""

    def __init__(self, R: np.ndarray, pas: np.ndarray, U: np.ndarray, edges: str = "all"):
        self.R = np.asarray(R, float)
        self.pas = np.asarray(pas, float)
        self.U = U
        m = len(self.pas)
        self.m = m
        self.cost_v = 100.0 * (self.pas[0] - self.pas)
        if m == 1:
            self.edges = []
        elif edges == "twin":   # twin-segment-only hull (sensitivity s1)
            self.edges = [(0, j) for j in range(1, m)]
        else:
            self.edges = list(itertools.combinations(range(m), 2))
        self._grid = None

    def eval_lams(self, L: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
        """L [G, m] mixture weights -> (hill [G], cost [G]); a point with no correct mass has Hill nan."""
        r = L @ self.R
        tot = r.sum(axis=1)
        cost = 100.0 * (self.pas[0] - L @ self.pas)
        hill = np.full(len(L), np.nan)
        ok = tot > 0
        if ok.any():
            Q = r[ok] / tot[ok, None]
            keep = Q.max(axis=0) > 0
            hill[ok] = pop_hill(Q[:, keep] if keep.any() else Q, self.U)
        return hill, cost

    def _edge_points(self, i: int, j: int, lam: np.ndarray) -> np.ndarray:
        L = np.zeros((len(lam), self.m))
        L[:, i] = 1.0 - lam
        L[:, j] += lam
        return L

    def grid(self) -> list[dict[str, Any]]:
        """Evaluate every edge on the 0.025 grid (cached)."""
        if self._grid is not None:
            return self._grid
        out = []
        if not self.edges:
            h, c = self.eval_lams(np.ones((1, 1)))
            out.append({"edge": (0, 0), "lam": np.array([0.0]), "hill": h, "cost": c})
        else:
            Ls = [self._edge_points(i, j, GRID) for i, j in self.edges]
            h, c = self.eval_lams(np.vstack(Ls))
            n = len(GRID)
            for e, (i, j) in enumerate(self.edges):
                out.append({"edge": (i, j), "lam": GRID, "hill": h[e * n:(e + 1) * n], "cost": c[e * n:(e + 1) * n]})
        self._grid = out
        return out

    def best_at_cost(self, c_max: float, refine: bool = True) -> dict[str, Any]:
        """max Hill over hull points with cost <= c_max (grid + exact boundary points + golden refinement)."""
        best = {"hill": -np.inf, "cost": None, "edge": None, "lam": None}
        bnd_L, bnd_meta = [], []
        for g in self.grid():
            feas = g["cost"] <= c_max + 1e-9
            if feas.any():
                hh = np.where(feas & np.isfinite(g["hill"]), g["hill"], -np.inf)
                k = int(np.argmax(hh))
                if hh[k] > best["hill"]:
                    best = {"hill": float(hh[k]), "cost": float(g["cost"][k]), "edge": g["edge"],
                            "lam": float(g["lam"][k])}
            if g["edge"] != (0, 0):
                i, j = g["edge"]
                ci, cj = self.cost_v[i], self.cost_v[j]
                if (ci - c_max) * (cj - c_max) < 0:  # boundary crossing inside the edge
                    lam_b = (c_max - ci) / (cj - ci)
                    bnd_L.append(self._edge_points(i, j, np.array([lam_b]))[0])
                    bnd_meta.append((g["edge"], lam_b))
        if bnd_L:
            hb, cb = self.eval_lams(np.array(bnd_L))
            for (e, lb), h_, c_ in zip(bnd_meta, hb, cb):
                if np.isfinite(h_) and h_ > best["hill"]:
                    best = {"hill": float(h_), "cost": float(c_), "edge": e, "lam": float(lb)}
        if refine and best["edge"] not in (None, (0, 0)):
            best = self._golden(best, c_max)
        if not np.isfinite(best["hill"]):
            best["hill"] = None
        return best

    def _golden(self, best: dict[str, Any], c_max: float, iters: int = 14) -> dict[str, Any]:
        i, j = best["edge"]
        ci, cj = self.cost_v[i], self.cost_v[j]
        lo, hi = max(0.0, best["lam"] - 0.025), min(1.0, best["lam"] + 0.025)
        # restrict to the feasible part of the edge
        if cj != ci:
            lam_b = (c_max - ci) / (cj - ci)
            if cj > ci:
                hi = min(hi, lam_b)
            else:
                lo = max(lo, lam_b)
        elif ci > c_max + 1e-9:
            return best
        if hi <= lo:
            return best
        gr = (math.sqrt(5) - 1) / 2

        def f(x):
            h, c = self.eval_lams(self._edge_points(i, j, np.array([x])))
            return (h[0] if np.isfinite(h[0]) else -np.inf), c[0]

        a, b = lo, hi
        x1, x2 = b - gr * (b - a), a + gr * (b - a)
        f1, f2 = f(x1)[0], f(x2)[0]
        for _ in range(iters):
            if f1 > f2:
                b, x2, f2 = x2, x1, f1
                x1 = b - gr * (b - a)
                f1 = f(x1)[0]
            else:
                a, x1, f1 = x1, x2, f2
                x2 = a + gr * (b - a)
                f2 = f(x2)[0]
        xm = x1 if f1 > f2 else x2
        hm, cm = f(xm)
        if hm > best["hill"] and cm <= c_max + 1e-9:
            return {"hill": float(hm), "cost": float(cm), "edge": (i, j), "lam": float(xm)}
        return best

    def price(self, thr: float, tol: float = 0.1, fast: bool = False) -> float | None:
        """Smallest c >= 0 with max Hill at cost <= c >= thr (None if never reached on the hull). fast=True (inside
        bootstrap replicates): the crossing between grid neighbours is linearly interpolated, not bisected."""
        cands = []
        for g in self.grid():
            h, c, lam = g["hill"], g["cost"], g["lam"]
            ok = np.isfinite(h) & (h >= thr)
            if not ok.any():
                continue
            k = np.where(ok)[0]
            cands.append(float(np.min(c[k])))
            if g["edge"] == (0, 0):
                continue
            # refine crossings between neighbours where hill crosses thr and the crossing point is cheaper
            for a in range(len(lam) - 1):
                if (ok[a] != ok[a + 1]) and np.isfinite(h[a]) and np.isfinite(h[a + 1]):
                    if fast:
                        t_ = (thr - h[a]) / (h[a + 1] - h[a]) if h[a + 1] != h[a] else 0.0
                        cands.append(float(c[a] + t_ * (c[a + 1] - c[a])))
                        continue
                    lo_l, hi_l = (lam[a], lam[a + 1]) if ok[a + 1] else (lam[a + 1], lam[a])
                    i, j = g["edge"]
                    for _ in range(8):  # bisection on lambda: lo_l fails, hi_l passes
                        mid = 0.5 * (lo_l + hi_l)
                        hm, _ = self.eval_lams(self._edge_points(i, j, np.array([mid])))
                        if np.isfinite(hm[0]) and hm[0] >= thr:
                            hi_l = mid
                        else:
                            lo_l = mid
                    _, cc = self.eval_lams(self._edge_points(i, j, np.array([hi_l])))
                    cands.append(float(cc[0]))
        if not cands:
            return None
        return max(0.0, round(min(cands), 1))

    def summary(self, refine: bool = True, with_price: bool = True) -> dict[str, Any]:
        b5 = self.best_at_cost(5.0, refine=refine)
        out = {"H5": b5["hill"], "H5_cost": b5["cost"], "H5_edge": b5["edge"], "H5_lam": b5["lam"]}
        if with_price:
            out["P15"] = self.price(1.5, fast=not refine)
            out["P20"] = self.price(2.0, fast=not refine)
        return out


def slsqp_check(h: Hull, c_max: float = 5.0, n_starts: int = 6, seed: int = 0) -> float | None:
    """Full-simplex multistart SLSQP over 3-vertex faces; returns the best Hill found (point-estimate check)."""
    from scipy.optimize import minimize

    if h.m < 3:
        return None
    rng = np.random.default_rng(seed)
    best = -np.inf
    for tri in itertools.combinations(range(h.m), 3):
        for _ in range(n_starts):
            x0 = rng.dirichlet(np.ones(3))

            def f(x, tri=tri):
                L = np.zeros((1, h.m))
                L[0, list(tri)] = np.clip(x, 0, None) / max(np.clip(x, 0, None).sum(), 1e-12)
                v, _ = h.eval_lams(L)
                return -(v[0] if np.isfinite(v[0]) else 0.0)

            def con(x, tri=tri):
                L = np.zeros((1, h.m))
                L[0, list(tri)] = np.clip(x, 0, None) / max(np.clip(x, 0, None).sum(), 1e-12)
                return c_max - float(100.0 * (h.pas[0] - L @ h.pas))

            r = minimize(f, x0, method="SLSQP", bounds=[(0, 1)] * 3,
                         constraints=[{"type": "eq", "fun": lambda x: x.sum() - 1}, {"type": "ineq", "fun": con}],
                         options={"maxiter": 60})
            if r.success or True:
                x = np.clip(r.x, 0, None)
                if con(x) >= -1e-6:
                    best = max(best, -f(x))
    return float(best) if np.isfinite(best) else None


def dial_interp_h5(points: list[tuple[float, float]], c_max: float = 5.0) -> float | None:
    """Sensitivity s3: piecewise-linear Hill along the dial (points ordered by setting strength, starting at the
    reference): max interpolated Hill at cost <= c_max."""
    best = None
    pts = [(c, h) for c, h in points if h is not None and np.isfinite(h)]
    for (c0, h0), (c1, h1) in zip(pts[:-1], pts[1:]):
        for c, hv in ((c0, h0), (c1, h1)):
            if c <= c_max and (best is None or hv > best):
                best = hv
        if (c0 - c_max) * (c1 - c_max) < 0:
            t = (c_max - c0) / (c1 - c0)
            hv = h0 + t * (h1 - h0)
            if best is None or hv > best:
                best = hv
    if len(pts) == 1 and pts[0][0] <= c_max:
        best = pts[0][1]
    return best


# ------------------------------------------------------------------------------------------------ unit tests
def unit_tests() -> dict[str, Any]:
    U = fixed_uniforms(1000)
    res = {}
    # (a) twin all default, setting all alternative, equal pass -> max at lambda 0.5, Hill = rarefied (0.5, 0.5)
    h = Hull(np.array([[1.0, 0.0], [0.0, 1.0]]), np.array([1.0, 1.0]), U)
    s = h.summary()
    ref = pop_hill(np.array([[0.5, 0.5]]), U)[0]
    res["a_lam"] = s["H5_lam"]
    res["a_pass"] = abs(s["H5_lam"] - 0.5) < 0.03 and abs(s["H5"] - ref) < 0.01 and abs(s["H5_cost"]) < 1e-9
    # (b) the same with a 10-point pass cost -> optimum on the cost = 5 boundary
    h = Hull(np.array([[1.0, 0.0], [0.0, 0.9]]), np.array([1.0, 0.9]), U)
    s = h.summary()
    res["b_cost"] = s["H5_cost"]
    res["b_pass"] = abs(s["H5_cost"] - 5.0) < 0.05
    # (c) a knob identical to the twin -> H5 = the twin's Hill
    R0 = np.array([0.6, 0.3, 0.1])
    h = Hull(np.vstack([R0, R0]), np.array([1.0, 1.0]), U)
    s = h.summary()
    res["c_pass"] = abs(s["H5"] - pop_hill(R0[None] / R0.sum(), U)[0]) < 1e-9
    # (d) adding a setting never decreases H5
    rng = np.random.default_rng(3)
    ok = True
    for _ in range(20):
        R = rng.dirichlet(np.ones(4), size=3) * rng.uniform(0.6, 1.0, size=(3, 1))
        pas = R.sum(1)
        a = Hull(R[:2], pas[:2], U).summary(with_price=False)["H5"]
        b = Hull(R, pas, U).summary(with_price=False)["H5"]
        ok &= (b is not None and a is not None and b >= a - 1e-9)
    res["d_pass"] = bool(ok)
    # edge search vs full-simplex SLSQP on random 5-vertex cases (point-estimate check)
    gaps = []
    U2 = fixed_uniforms(300)
    for t in range(4):
        R = rng.dirichlet(np.ones(4) * 0.5, size=5) * rng.uniform(0.85, 1.0, size=(5, 1))
        hh = Hull(R, R.sum(1), U2)
        e = hh.summary(with_price=False)["H5"]
        q = slsqp_check(hh, n_starts=2, seed=t)
        gaps.append((q - e) if (q is not None and e is not None) else 0.0)
    res["slsqp_minus_edge_max"] = float(max(gaps))
    res["slsqp_pass"] = res["slsqp_minus_edge_max"] <= 0.02
    res["all_pass"] = all(res[k] for k in ("a_pass", "b_pass", "c_pass", "d_pass", "slsqp_pass"))
    return res

### Hull unit tests

Tests (a)-(d) check the hull on synthetic cases:
- (a) a 50/50 mixture of two pure arms is optimal;
- (b) a pass cost moves the optimum onto the cost = 5 boundary;
- (c) a knob identical to the twin prices at the twin's Hill;
- (d) adding a setting never lowers H5.

The SLSQP line compares the edge search with a full-simplex optimiser on random 5-vertex hulls. The full run reported a
gap of 0.19 on these *synthetic* cases. That is why `point_prices` below also runs a 3-vertex-face check
(`triangle_minus_edge`) on the real PN, TEMP and VETO_ITER hulls. The cell also checks that the vectorised `pop_hill`
equals its reference row loop.

In [ ]:
import warnings

if RUN_HULL_UNIT_TESTS:
    t0 = time.time()
    with warnings.catch_warnings():
        # slsqp_check calls float() on a 1-element array: a harmless NumPy >= 1.25 DeprecationWarning
        warnings.simplefilter("ignore", DeprecationWarning)
        hull_ut = {k: (v.item() if hasattr(v, "item") else v) for k, v in unit_tests().items()}  # numpy -> Python
    print("this run:", hull_ut)
    print("full run:", meta["original_results"]["hull_unit_tests"])
    Qt = np.random.default_rng(0).dirichlet(np.ones(4), size=8)
    print("pop_hill == pop_hill_loop on 8 random compositions:",
          bool(np.allclose(pop_hill(Qt, fixed_uniforms(100)), pop_hill_loop(Qt, fixed_uniforms(100)))))
    print(f"({time.time() - t0:.1f}s)")

## Knob classes, task data and knob rows (`src/analysis/core.py`, part 1)

**Knob classes.**
- `CLASSES` groups the arms into knob classes, each with its settings in dial order and its reference arm.
- `NA_PRIMARY` holds the non-activation (NA) classes of the primary rule. PN is compared against these.
- MINP and VS were pre-registered sensitivity knobs (s2), and PN alpha 8 / PNPLAIN are descriptive.

**Correct programs.** A program is *correct* if it passed the hidden tests and its label is not `invalid`. Its *idea
category* is its mechanism label. Before that, the committed `other` mapping is applied: the novelty audit found that
all 12 correct `other` programs were labeller misses, so they map back to catalogue labels.

**TaskData.** It turns the rows of each arm into arrays: category index (-1 if incorrect), pass flag, template stratum
and CRN key.

**knob_rows.** It writes one Step-0 row per (task, arm). These rows were the merge-ready output for the next
evaluation.

**Notebook change:** `n_boot=2000` -> `KNOB_ROW_N_BOOT`.

In [ ]:
DEPTH = 25
E6 = ("T03", "T05", "T06", "T10", "T1", "T2")
# knob classes: name -> (settings in dial order, reference arm). VETO classes use the reused rows' own twin.
CLASSES: dict[str, dict[str, Any]] = {
    "PN": {"settings": ["PN_a0.5", "PN_a1", "PN_a2", "PN_a4"], "ref": "TWIN", "na": False},
    "TEMP": {"settings": ["T1.3", "T1.6"], "ref": "TWIN", "na": True},
    "TOPK10": {"settings": ["TOPK10"], "ref": "TWIN", "na": True},
    "NONCE": {"settings": ["NONCE"], "ref": "TWIN", "na": True},
    "NOVELTY": {"settings": ["NOVELTY"], "ref": "TWIN", "na": True},
    "SEMSEED": {"settings": ["SEMSEED"], "ref": "TWIN", "na": True},
    "VETO_1SHOT": {"settings": ["VETO_1SHOT"], "ref": "REUSED_TWIN", "na": True, "reused": True},
    "VETO_PERSIST": {"settings": ["VETO_PERSIST"], "ref": "REUSED_TWIN", "na": True, "reused": True},
    "VETO_ITER": {"settings": ["VETO_ITER1", "VETO_ITER2", "VETO_ITER3"], "ref": "REUSED_TWIN", "na": True,
                  "reused": True},
    # pre-registered sensitivity s2 (added to NA only in s2)
    "MINP": {"settings": ["MINP_T1.6", "MINP_T2.0"], "ref": "TWIN", "na": False, "s2": True},
    "VS": {"settings": ["VS"], "ref": "TWIN", "na": False, "s2": True},
    # descriptive (never in the C1 rule)
    "PN_EXTRA": {"settings": ["PN_a0.5", "PN_a1", "PN_a2", "PN_a4", "PN_a8"], "ref": "TWIN", "na": False,
                 "descriptive": True},
    "PNPLAIN": {"settings": ["PNPLAIN_a1"], "ref": "PLAIN", "na": False, "descriptive": True},
}
NA_PRIMARY = [k for k, v in CLASSES.items() if v["na"]]
NA_S2 = NA_PRIMARY + ["MINP", "VS"]


def idea_label(r: dict[str, Any], other_map: dict[str, str] | None, convention: str = "mapped") -> str | None:
    """Hill category of a CORRECT program (None = excluded from Hill counts)."""
    lab = r["label"]
    if lab == "other":
        if convention == "exclude":
            return None
        if convention == "one" or other_map is None:
            return "other"
        m = other_map.get(r.get("code_sha256") or "", "other")
        return None if m == "DEGENERATE" else m
    return lab


def is_correct(r: dict[str, Any]) -> bool:
    return bool(r.get("passed")) and r.get("label") not in ("invalid", None)


class TaskData:
    """Per task: for each arm, arrays (category index or -1, passed, stratum index, slot key)."""

    def __init__(self, task: str, rows: list[dict[str, Any]], other_map: dict[str, str] | None,
                 convention: str = "mapped"):
        self.task = task
        cats: set[str] = set()
        by_arm: dict[str, list[dict[str, Any]]] = defaultdict(list)
        for r in rows:
            by_arm[r["arm"]].append(r)
            if is_correct(r):
                c = idea_label(r, other_map, convention)
                if c is not None:
                    cats.add(c)
        self.cats = sorted(cats)
        ci = {c: i for i, c in enumerate(self.cats)}
        self.arms: dict[str, dict[str, Any]] = {}
        for arm, rs in by_arm.items():
            rs = sorted(rs, key=lambda r: (str(r.get("template")), str(r.get("slot"))))
            cat = np.array([ci[c] if (is_correct(r) and (c := idea_label(r, other_map, convention)) is not None)
                            else -1 for r in rs], dtype=int)
            pas = np.array([is_correct(r) for r in rs], dtype=bool)
            strata = [str(r.get("template")) for r in rs]
            su = sorted(set(strata))
            sidx = np.array([su.index(s) for s in strata], dtype=int)
            keys = [f"{r.get('template')}|{r.get('slot')}" for r in rs]
            self.arms[arm] = {"cat": cat, "pass": pas, "stratum": sidx, "n_strata": len(su), "keys": keys,
                              "engine": rs[0].get("engine"), "n": len(rs)}

    def vertex(self, arm: str, idx: np.ndarray | None = None) -> tuple[np.ndarray, float, int]:
        a = self.arms[arm]
        cat = a["cat"] if idx is None else a["cat"][idx]
        pas = a["pass"] if idx is None else a["pass"][idx]
        n = len(cat)
        r = np.bincount(cat[cat >= 0], minlength=len(self.cats)).astype(float) / max(n, 1)
        return r, float(pas.mean()) if n else float("nan"), n

    def counts(self, arm: str) -> Counter:
        a = self.arms[arm]
        return Counter(self.cats[c] for c in a["cat"] if c >= 0)


def class_vertices(td: TaskData, cls: str, ref_override: str | None = None) -> list[str] | None:
    spec = CLASSES[cls]
    ref = ref_override or spec["ref"]
    if ref not in td.arms:
        return None
    sets = [s for s in spec["settings"] if s in td.arms]
    if not sets:
        return None
    return [ref] + sets


def class_price(td: TaskData, cls: str, U: np.ndarray, idx_by_arm: dict[str, np.ndarray] | None = None,
                edges: str = "all", with_price: bool = True, refine: bool = True) -> dict[str, Any] | None:
    arms = class_vertices(td, cls)
    if arms is None:
        return None
    R, P = [], []
    for a in arms:
        r, p, _ = td.vertex(a, None if idx_by_arm is None else idx_by_arm.get(a))
        R.append(r)
        P.append(p)
    h = Hull(np.array(R), np.array(P), U, edges=edges)
    s = h.summary(refine=refine, with_price=with_price)
    s["arms"] = arms
    s["vertex_cost"] = h.cost_v.round(3).tolist()
    return s


def step0_hill(counts: Counter) -> float:
    c = [v for _, v in counts.most_common()]
    n = sum(c)
    if n == 0:
        return float("nan")
    if n < DEPTH:
        return rarefied_hill1(c, n, 1000, np.random.default_rng(0)) if n > 1 else 1.0
    return rarefied_hill1(c, DEPTH, 1000, np.random.default_rng(0))


def knob_rows(tds: dict[str, TaskData], ref_of: dict[str, str]) -> list[dict[str, Any]]:
    """Plan 6.1: one row per (task, arm), Step-0 statistic with its 90% interval."""
    out = []
    for t, td in tds.items():
        for arm, a in td.arms.items():
            cnt = td.counts(arm)
            n_c = sum(cnt.values())
            ref = ref_of.get(arm, "TWIN")
            p_ref = float(td.arms[ref]["pass"].mean()) if ref in td.arms else None
            p = float(a["pass"].mean())
            cc = [v for _, v in cnt.most_common()]
            ci = boot_rarefied_hill1(cc, DEPTH, n_boot=KNOB_ROW_N_BOOT, n_sub=200, rng=np.random.default_rng(1),
                                     conf=0.90) if n_c >= DEPTH else (float("nan"), float("nan"))
            out.append({"task": t, "arm": arm, "knob_class": knob_of(arm), "engine": a["engine"],
                        "reference_arm": ref, "n": a["n"], "n_correct": int(a["pass"].sum()),
                        "n_correct_hill": n_c, "pass": p,
                        "pass_cost_pts": (100 * (p_ref - p)) if p_ref is not None else None,
                        "correct_comp": dict(cnt.most_common()), "hill25_step0": step0_hill(cnt),
                        "ci90": list(ci), "depth_flag": n_c < DEPTH})
    return out


def knob_of(arm: str) -> str:
    for k, v in CLASSES.items():
        if k in ("PN_EXTRA",):
            continue
        if arm in v["settings"]:
            return k
    if arm in ("TWIN", "PLAIN", "REUSED_TWIN"):
        return "REF"
    if arm == "PN_a8":
        return "PN_EXTRA"
    return "OTHER"

## Bootstrap and the pre-registered C1 rule (`src/analysis/core.py`, part 2)

**Bootstrap.** In each replicate, programs are resampled with replacement within each arm, stratified by template. The
two alternative schemes are cluster resampling of templates (s4) and paired resampling of CRN slots across arms (s5).
Each replicate then **re-optimises every class's hull**, so the knob's best setting is chosen *inside* the replicate.
This charges a knob for its own selection noise.

**C1 rule (`c1_decide`).**
- **Per-task margin:** `D_t = H5_PN - max over NA classes of H5`.
- **Counts:** count1 = #{H5_PN >= 1.5} and count2 = #{H5_PN > H5_best}.
- **Pooled margin:** D_bar is the mean of D_t. Its CI comes from a two-level bootstrap that resamples tasks, then
  draws one replicate D_t per drawn task.
- **Verdict:** CONFIRMED needs count1 >= 3, count2 >= 4 and CI_low > 0. DISCONFIRMED needs count1 <= 1, or the best NA
  knob winning on >= 4 tasks. Everything else is PARTIAL.

**Notebook change:** the original ran the seeded bootstrap jobs in a spawn-based `ProcessPoolExecutor`. Spawned workers
cannot import functions defined in a notebook, so the same jobs run serially here.

In [ ]:
def resample_idx(td: TaskData, rng: np.random.Generator, scheme: str = "stratified") -> dict[str, np.ndarray]:
    """Programs resampled with replacement within each arm, stratified by template (primary); template-cluster (s4:
    resample templates, then programs within them); slot-paired (s5: resample (template, slot) keys jointly across
    every arm of this artifact; reused rows resample within their arm)."""
    out: dict[str, np.ndarray] = {}
    if scheme == "slot_paired":
        keyset = sorted({k for arm, a in td.arms.items() if not arm.startswith(("VETO", "REUSED")) for k in a["keys"]})
        draw = rng.choice(len(keyset), size=len(keyset), replace=True)
        drawn = [keyset[i] for i in draw]
    if scheme == "template_cluster":
        allw = sorted({k.split("|")[0] for a in td.arms.values() for k in a["keys"]})
        wdraw = [allw[i] for i in rng.choice(len(allw), size=len(allw), replace=True)]
    for arm, a in td.arms.items():
        n = a["n"]
        if scheme == "slot_paired" and not arm.startswith(("VETO", "REUSED")):
            pos = {k: i for i, k in enumerate(a["keys"])}
            idx = [pos[k] for k in drawn if k in pos]
            if len(idx) < max(1, n // 2):
                idx = list(rng.integers(0, n, size=n))
            out[arm] = np.array(idx, dtype=int)
            continue
        if scheme == "template_cluster":
            ws = [k.split("|")[0] for k in a["keys"]]
            groups = defaultdict(list)
            for i, w in enumerate(ws):
                groups[w].append(i)
            idx = []
            for w in wdraw:
                g = groups.get(w)
                if g:
                    idx.extend(rng.choice(g, size=len(g), replace=True).tolist())
            if not idx:
                idx = list(rng.integers(0, n, size=n))
            out[arm] = np.array(idx, dtype=int)
            continue
        idx = np.empty(n, dtype=int)
        pos = 0
        for s in range(a["n_strata"]):
            members = np.where(a["stratum"] == s)[0]
            k = len(members)
            idx[pos:pos + k] = members[rng.integers(0, k, size=k)]
            pos += k
        out[arm] = idx
    return out


def task_replicates(args: tuple) -> dict[str, Any]:
    """Worker: B bootstrap replicates of one task -> per-class H5/P15/P20 per replicate."""
    td, classes, B, seed, scheme, n_sub, edges = args
    U = fixed_uniforms(n_sub)
    rng = np.random.default_rng(seed)
    res: dict[str, dict[str, list]] = {c: {"H5": [], "P15": [], "P20": []} for c in classes}
    for b in range(B):
        idx = resample_idx(td, rng, scheme)
        for c in classes:
            s = class_price(td, c, U, idx, edges=edges, with_price=True, refine=False)
            for k in ("H5", "P15", "P20"):
                res[c][k].append(None if s is None else s[k])
    return {"task": td.task, "res": res}


def run_bootstrap(tds: dict[str, TaskData], classes: list[str], B: int, seed: int, scheme: str = "stratified",
                  n_sub: int = 200, edges: str = "all", workers: int = 6) -> dict[str, dict[str, Any]]:
    jobs = []
    chunk = max(1, B // workers)
    for t, td in tds.items():
        start = 0
        part = 0
        while start < B:
            nb = min(chunk, B - start)
            jobs.append((td, [c for c in classes if class_vertices(td, c) is not None], nb,
                         seed + 1000 * part + zlib.crc32(t.encode()) % 997, scheme, n_sub, edges))
            start += nb
            part += 1
    out: dict[str, dict[str, Any]] = {t: {} for t in tds}
    # notebook change: the original ran `jobs` in ProcessPoolExecutor(max_workers=workers, mp_context=spawn); the same
    # seeded jobs run serially here (spawned workers cannot import functions defined in a notebook)
    for r in map(task_replicates, jobs):
        for c, d in r["res"].items():
            o = out[r["task"]].setdefault(c, {"H5": [], "P15": [], "P20": []})
            for k in ("H5", "P15", "P20"):
                o[k].extend(d[k])
    return out


def pct_ci(x: list, lo: float = 2.5, hi: float = 97.5) -> list[float | None]:
    v = np.array([np.nan if a is None else a for a in x], dtype=float)
    v = v[np.isfinite(v)]
    if len(v) == 0:
        return [None, None]
    return [float(np.percentile(v, lo)), float(np.percentile(v, hi))]


# ------------------------------------------------------------------------------------------------ C1
def best_na(h5: dict[str, float | None], na: list[str]) -> tuple[str | None, float | None]:
    cand = [(c, h5.get(c)) for c in na if h5.get(c) is not None and np.isfinite(h5.get(c))]
    if not cand:
        return None, None
    c, v = max(cand, key=lambda x: x[1])
    return c, v


def c1_decide(point: dict[str, dict[str, float | None]], boot: dict[str, dict[str, Any]], tasks: tuple[str, ...],
              na: list[str], B: int, seed: int) -> dict[str, Any]:
    """Pre-registered rule: CONFIRMED iff count1 >= 3 AND count2 >= 4 AND CI_low(D_bar) > 0; DISCONFIRMED iff
    count1 <= 1 OR #{t: H5_best > H5_PN} >= 4; otherwise PARTIAL."""
    per = {}
    D = {}
    for t in tasks:
        h5 = point.get(t, {})
        pn = h5.get("PN")
        bc, bv = best_na(h5, na)
        if pn is None or bv is None:
            per[t] = {"status": "UNDECIDABLE", "H5_PN": pn, "best_na": bc, "H5_best": bv}
            continue
        D[t] = pn - bv
        # per-task bootstrap probability P(H5_PN > H5_best), best chosen inside each replicate
        bt = boot.get(t, {})
        nrep = len(bt.get("PN", {}).get("H5", []))
        wins, dts = 0, []
        for b in range(nrep):
            hp = bt["PN"]["H5"][b]
            hb = [bt[c]["H5"][b] for c in na if c in bt and bt[c]["H5"][b] is not None]
            if hp is None or not hb:
                dts.append(np.nan)
                continue
            dts.append(hp - max(hb))
            wins += hp > max(hb)
        per[t] = {"H5_PN": pn, "best_na": bc, "H5_best": bv, "D_t": D[t], "P_boot_PN_gt_best": wins / max(nrep, 1),
                  "D_t_boot_ci95": pct_ci(dts), "D_t_boot": dts}
    decided = [t for t in tasks if t in D]
    count1 = sum(1 for t in decided if per[t]["H5_PN"] >= 1.5)
    count2 = sum(1 for t in decided if per[t]["H5_PN"] > per[t]["H5_best"])
    n_best_wins = sum(1 for t in decided if per[t]["H5_best"] > per[t]["H5_PN"])
    D_bar = float(np.mean([D[t] for t in decided])) if decided else None
    rng = np.random.default_rng(seed)
    Dstar = []
    for _ in range(B):
        draw = rng.choice(decided, size=len(decided), replace=True)
        vals = []
        for t in draw:
            dd = np.array(per[t]["D_t_boot"], dtype=float)
            dd = dd[np.isfinite(dd)]
            vals.append(dd[rng.integers(0, len(dd))] if len(dd) else D[t])
        Dstar.append(float(np.mean(vals)))
    Dstar = np.array(Dstar)
    ci = [float(np.percentile(Dstar, 2.5)), float(np.percentile(Dstar, 97.5))] if len(Dstar) else [None, None]
    p_c1 = float((1 + (Dstar <= 0).sum()) / (len(Dstar) + 1)) if len(Dstar) else None
    from scipy.stats import binomtest

    n_pos = sum(1 for t in decided if D[t] > 0)
    n_nonzero = sum(1 for t in decided if D[t] != 0)
    sign_p = float(binomtest(n_pos, n_nonzero, 0.5, alternative="greater").pvalue) if n_nonzero else None
    if count1 >= 3 and count2 >= 4 and ci[0] is not None and ci[0] > 0:
        verdict, failed = "CONFIRMED", []
    elif count1 <= 1 or n_best_wins >= 4:
        verdict = "DISCONFIRMED"
        failed = [x for x, ok in (("count1<=1", count1 <= 1), ("best>PN on >=4 tasks", n_best_wins >= 4)) if ok]
    else:
        verdict = "PARTIAL"
        failed = [x for x, ok in (("count1>=3", count1 >= 3), ("count2>=4", count2 >= 4),
                                  ("CI_low>0", ci[0] is not None and ci[0] > 0)) if not ok]
    for t in per:
        per[t].pop("D_t_boot", None)
    return {"verdict": verdict, "failed_thresholds": failed, "count1": count1, "count2": count2,
            "n_tasks_best_beats_PN": n_best_wins, "D_bar": D_bar, "D_bar_ci95": ci, "p_C1_one_sided": p_c1,
            "sign_test_p_one_sided": sign_p, "n_positive_D": n_pos, "decided_tasks": decided, "per_task": per,
            "undecidable_tasks": [t for t in tasks if t not in D]}

## The analysis driver (`src/analysis/run_all.py`)

**Helpers in this cell.**
- `task_rows` adds the reused iteration-3 rows to a task. T1 has no iteration-3 twin, so its vetoes are priced against
  this run's TWIN (a flagged deviation).
- `point_prices` computes, per task and class, H5, P15 and P20, the s1 / s3 / s8 alternatives and the 3-vertex-face
  check.
- `boot_alt` bootstraps the s3 / s8 alternatives.
- `c1_all` evaluates the primary rule and the sensitivities.

**Sensitivities.** Each one re-evaluates the full rule:
- s1: twin-segment hull only;
- s3: dial interpolation;
- s4: template-cluster bootstrap;
- s5: slot-paired bootstrap;
- s6: `other` excluded / one category;
- s7: reused vetoes excluded;
- s8: Step-0 vertices only.

**Notebook changes:**
- the bootstrap cache under `results/boot_cache/` is dropped (it only saved time and gave the same replicates);
- the literals 1000 / 2000 become the config variables;
- the process pools run serially;
- s2 / s2b are skipped, because they add the MINP and VS arms, which are not in the 99-cell demo subset.

In [ ]:
PRICE_CLASSES = ["PN", "TEMP", "TOPK10", "NONCE", "NOVELTY", "SEMSEED", "VETO_1SHOT", "VETO_PERSIST", "VETO_ITER",
                 "MINP", "VS", "PN_EXTRA", "PNPLAIN"]
BOOT_CLASSES = ["PN", "TEMP", "TOPK10", "NONCE", "NOVELTY", "SEMSEED", "VETO_1SHOT", "VETO_PERSIST", "VETO_ITER",
                "MINP", "VS"]


def cached_bootstrap(tds: dict[str, TaskData], classes: list[str], B: int, seed: int, scheme: str = "stratified",
                     edges: str = "all", workers: int = 6, n_sub: int = 200) -> dict[str, Any]:
    """run_bootstrap per task (the original cached each task's replicates under results/boot_cache/; speed only, same
    replicates -- the notebook computes them directly)."""
    out = run_bootstrap(tds, classes, B, seed, scheme=scheme, n_sub=n_sub, edges=edges, workers=workers)
    logger.info(f"bootstrap {scheme}/{edges}: 0 tasks from cache, {len(tds)} computed")
    return out


def task_rows(samples: list[dict[str, Any]], reused: list[dict[str, Any]], task: str,
              include_reused: bool = True) -> list[dict[str, Any]]:
    rows = [r for r in samples if r["task"] == task]
    if include_reused:
        rr = [r for r in reused if r["task"] == task]
        if not any(r["arm"] == "REUSED_TWIN" for r in rr):
            # T1: no iteration-3 twin -> price the reused vetoes against this artifact's TWIN (flagged)
            rr += [dict(r, arm="REUSED_TWIN", sample_id="reftwin|" + r["sample_id"]) for r in rows
                   if r["arm"] == "TWIN"]
        rows += rr
    return rows


def triangle_check(td: TaskData, cls: str, U: np.ndarray, step: float = 0.05) -> float | None:
    """Best Hill at cost <= 5 over every 3-vertex face (grid step 0.05) minus the edge-search H5."""
    arms = class_vertices(td, cls)
    if arms is None or len(arms) < 3:
        return None
    R = np.array([td.vertex(a)[0] for a in arms])
    P = np.array([td.vertex(a)[1] for a in arms])
    h = Hull(R, P, U)
    g = np.arange(0, 1 + 1e-9, step)
    best = -np.inf
    for tri in itertools.combinations(range(len(arms)), 3):
        L = []
        for a in g:
            for b in g:
                if a + b <= 1 + 1e-9:
                    v = np.zeros(len(arms))
                    v[tri[0]], v[tri[1]], v[tri[2]] = 1 - a - b, a, b
                    L.append(v)
        hh, cc = h.eval_lams(np.array(L))
        ok = (cc <= 5 + 1e-9) & np.isfinite(hh)
        if ok.any():
            best = max(best, float(hh[ok].max()))
    e = h.summary(with_price=False)["H5"]
    return (best - e) if (np.isfinite(best) and e is not None) else None


def alt_h5(td: TaskData, cls: str, kind: str) -> float | None:
    """s8: Step-0 hypergeometric vertices only (no mixing): max over {ref} u settings with cost <= 5.
    s3: dial interpolation of the Step-0 vertex Hill along the dial."""
    arms = class_vertices(td, cls)
    if arms is None:
        return None
    pts = []
    p0 = td.vertex(arms[0])[1]
    for a in arms:
        cnt = td.counts(a)
        h = step0_hill(cnt) if sum(cnt.values()) > 0 else float("nan")
        c = 100 * (p0 - td.vertex(a)[1])
        pts.append((c, h))
    if kind == "s8":
        ok = [h for c, h in pts if c <= 5 and np.isfinite(h)]
        return max(ok) if ok else None
    return dial_interp_h5(pts)


def point_prices(tds: dict[str, TaskData]) -> dict[str, Any]:
    U = fixed_uniforms(N_SUB_POINT)
    out: dict[str, Any] = {}
    for t, td in tds.items():
        out[t] = {}
        for c in PRICE_CLASSES:
            s = class_price(td, c, U, refine=True)
            if s is None:
                continue
            s1 = class_price(td, c, U, edges="twin", refine=True, with_price=False)
            s["H5_s1_twin_segment"] = s1["H5"] if s1 else None
            s["H5_s3_dial"] = alt_h5(td, c, "s3")
            s["H5_s8_step0_vertices"] = alt_h5(td, c, "s8")
            s["triangle_minus_edge"] = triangle_check(td, c, fixed_uniforms(300)) if c in ("PN", "TEMP",
                                                                                         "VETO_ITER") else None
            # vertex gap: population-rarefied (hull) vs Step-0 hypergeometric at each vertex
            gaps = []
            for a in s["arms"]:
                r, p, n = td.vertex(a)
                if r.sum() > 0 and sum(td.counts(a).values()) >= 25:
                    hp = Hull(r[None], np.array([p]), U).eval_lams(np.ones((1, 1)))[0][0]
                    gaps.append(float(step0_hill(td.counts(a)) - hp))
            s["max_vertex_gap_step0_minus_pop"] = max(gaps, key=abs) if gaps else None
            out[t][c] = s
    return out


def c1_all(point: dict[str, Any], boot: dict[str, Any], extra_boots: dict[str, Any]) -> dict[str, Any]:
    def h5_of(src: dict[str, Any], key: str = "H5") -> dict[str, dict[str, float | None]]:
        return {t: {c: v.get(key) for c, v in src[t].items()} for t in src}

    B = N_BOOT_C1  # notebook: the original passed the literal 2000 to every c1_decide call
    res: dict[str, Any] = {"utc": utc_now(), "rule": "predictions/c1_rule.json"}
    res["primary"] = c1_decide(h5_of(point), boot, E6, NA_PRIMARY, B, BOOT_SEED)
    res["s1_twin_segment_hull"] = c1_decide(h5_of(point, "H5_s1_twin_segment"), extra_boots.get("s1", {}), E6,
                                            NA_PRIMARY, B, BOOT_SEED + 1)
    # notebook: s2 / s2b add the MINP and VS arms to the comparison set; those arms are not in the demo subset, so here
    # they would only repeat the primary rule (full run: both PARTIAL, see the sensitivity table at the end)
    # res["s2_minp_vs_added"] = c1_decide(h5_of(point), boot, E6, NA_S2, 2000, BOOT_SEED + 2)
    # res["s2b_minp_only_added"] = c1_decide(h5_of(point), boot, E6, NA_PRIMARY + ["MINP"], 2000, BOOT_SEED + 22)
    res["s3_dial_interpolation"] = c1_decide(h5_of(point, "H5_s3_dial"), extra_boots.get("s3", {}), E6, NA_PRIMARY,
                                             B, BOOT_SEED + 3)
    res["s4_template_cluster_bootstrap"] = c1_decide(h5_of(point), extra_boots.get("s4", {}), E6, NA_PRIMARY, B,
                                                     BOOT_SEED + 4)
    res["s5_slot_paired_bootstrap"] = c1_decide(h5_of(point), extra_boots.get("s5", {}), E6, NA_PRIMARY, B,
                                                BOOT_SEED + 5)
    for conv in ("exclude", "one"):
        if f"s6_{conv}" in extra_boots:
            res[f"s6_other_{conv}"] = c1_decide(extra_boots[f"s6_{conv}_point"], extra_boots[f"s6_{conv}"], E6,
                                                NA_PRIMARY, B, BOOT_SEED + 6)
    na7 = [c for c in NA_PRIMARY if not c.startswith("VETO")]
    res["s7_reused_vetoes_excluded"] = c1_decide(h5_of(point), boot, E6, na7, B, BOOT_SEED + 7)
    res["s8_step0_vertices_only"] = c1_decide(h5_of(point, "H5_s8_step0_vertices"), extra_boots.get("s8", {}), E6,
                                              NA_PRIMARY, B, BOOT_SEED + 8)
    return res


def boot_alt(tds: dict[str, TaskData], B: int, seed: int, kind: str, workers: int) -> dict[str, Any]:
    """Bootstrap of the s3/s8 alternative H5 (Step-0 vertices, n_sub 200 inside replicates)."""
    jobs = [(td, kind, B, seed + i) for i, td in enumerate(tds.values())]
    out = {}
    # notebook change: the original mapped `jobs` over a spawn-based ProcessPoolExecutor; here they run serially
    for t, r in zip(tds, map(_alt_worker, jobs)):
        out[t] = r
    return out


def _alt_worker(args: tuple) -> dict[str, Any]:
    dih = dial_interp_h5  # the original imported resample_idx and dial_interp_h5 from the analysis package here

    td, kind, B, seed = args
    rng = np.random.default_rng(seed)
    res: dict[str, dict[str, list]] = {}
    for b in range(B):
        idx = resample_idx(td, rng)
        for c in BOOT_CLASSES:
            arms = class_vertices(td, c)
            if arms is None:
                continue
            pts = []
            p0 = float(td.arms[arms[0]]["pass"][idx[arms[0]]].mean())
            for a in arms:
                cat = td.arms[a]["cat"][idx[a]]
                cnt = np.bincount(cat[cat >= 0], minlength=len(td.cats))
                cc = cnt[cnt > 0]
                n = int(cc.sum())
                if n == 0:
                    h = np.nan
                elif len(cc) == 1:
                    h = 1.0
                else:
                    d = min(25, n)
                    h = rarefied_hill1(cc.tolist(), d, 200, rng)
                pts.append((100 * (p0 - float(td.arms[a]["pass"][idx[a]].mean())), h))
            if kind == "s8":
                ok = [h for cst, h in pts if cst <= 5 and np.isfinite(h)]
                v = max(ok) if ok else None
            else:
                v = dih(pts)
            res.setdefault(c, {"H5": []})["H5"].append(v)
    return res

## Run the `analyze` stage

### 1. Labelled programs to per-task arrays

In the full run, `merge_samples()` joined `data/samples_raw.jsonl` with `data/labels.jsonl`, and the reused veto rows
came from `data/reused_rows.jsonl`. The demo file already holds the merged rows, grouped by (task, arm) cell, so this
cell flattens them back into the same two lists.

The idea categories of each task are printed below. They are the mechanism labels of all its correct programs, across
every arm.

In [ ]:
t_analysis = time.time()
samples, reused = [], []
for ex in cells:
    for r in ex["rows"]:
        row = dict(r, task=ex["task"], arm=ex["arm"], engine=ex["engine"])
        if ex["source"] == "this_run":
            row["sample_id"] = f"{ex['task']}|{ex['arm']}|{r['template']}|{r['slot']}"
            samples.append(row)
        else:
            row["sample_id"] = f"reused|{r['slot']}"
            reused.append(row)
omap = meta["other_map"]   # audit/other_map.json: code hash of a correct 'other' program -> catalogue label
logger.info(f"{len(samples)} labelled samples of this run, {len(reused)} reused iteration-3 rows")

tasks = sorted({r["task"] for r in samples})
tds = {t: TaskData(t, task_rows(samples, reused, t), omap) for t in tasks}
for t, td in tds.items():
    print(f"{t:4s} {len(td.arms):2d} arms | idea categories: {', '.join(td.cats)}")

### 2. Knob rows: Step-0 Hill per (task, arm)

The Step-0 Hill of each row is deterministic, with fixed seeds, so it must equal the full run's
`results/knob_rows.jsonl` exactly. The 90% interval matches only when `KNOB_ROW_N_BOOT = 2000`.

In [ ]:
t0 = time.time()
ref_of = {"PNPLAIN_a1": "PLAIN", "VETO_1SHOT": "REUSED_TWIN", "VETO_PERSIST": "REUSED_TWIN",
          "VETO_ITER1": "REUSED_TWIN", "VETO_ITER2": "REUSED_TWIN", "VETO_ITER3": "REUSED_TWIN"}
kr = knob_rows(tds, ref_of)
logger.info(f"knob rows: {len(kr)} ({time.time() - t0:.0f}s)")


def _same(a, b, tol=1e-9):
    a = None if a is None or (isinstance(a, float) and math.isnan(a)) else a
    b = None if b is None or (isinstance(b, float) and math.isnan(b)) else b
    return (a is None and b is None) or (a is not None and b is not None and abs(a - b) <= tol)


orig_kr = {(ex["task"], ex["arm"]): ex["original_knob_row"] for ex in cells}
chk = [_same(r["hill25_step0"], orig_kr[(r["task"], r["arm"])]["hill25_step0"]) for r in kr if (r["task"], r["arm"]) in orig_kr]
print(f"Step-0 Hill identical to the full run's knob rows in {sum(chk)}/{len(chk)} cells")
kr_df = pd.DataFrame(kr)
kr_df["hill25_step0"] = kr_df["hill25_step0"].round(3)
kr_df["ci90"] = kr_df["ci90"].map(lambda v: [None if x is None or not np.isfinite(x) else round(x, 2) for x in v])
kr_df[kr_df.task == "T03"][["task", "arm", "knob_class", "n", "pass", "pass_cost_pts", "correct_comp",
                            "hill25_step0", "ci90"]]

### 3. Point prices on the randomisation hull

For every task and knob class, the hull gives H5, P15 and P20. The s1 (twin segment), s3 (dial) and s8 (Step-0
vertices) alternatives are computed alongside. With `N_SUB_POINT = 1000`, every value equals the full run's
`results/prices.json`.

In [ ]:
t0 = time.time()
point = point_prices(tds)
logger.info(f"point prices done ({time.time() - t0:.0f}s)")
orig_prices = meta["original_results"]["prices"]
n_same = n_all = 0
for t in point:
    for c, s in point[t].items():
        o = orig_prices.get(t, {}).get(c)
        if o is None:
            continue
        for k in ("H5", "P15", "P20", "H5_s1_twin_segment", "H5_s3_dial", "H5_s8_step0_vertices"):
            n_all += 1
            n_same += _same(s[k], o[k])
print(f"point estimates identical to the full run: {n_same}/{n_all} "
      f"(N_SUB_POINT = {N_SUB_POINT}; 1000 reproduces the paper exactly)")
print("H5 per task (columns) and knob class (rows):")
pd.DataFrame({t: {c: s["H5"] for c, s in point[t].items()} for t in point}).round(2)

### 4. Primary bootstrap

Programs are resampled within each arm, stratified by template. In every replicate, all 11 classes are re-priced on
their hulls, with fast settings: no golden refinement, and linear crossing interpolation.

In [ ]:
t0 = time.time()
boot = cached_bootstrap(tds, BOOT_CLASSES, N_BOOT, BOOT_SEED, workers=WORKERS, n_sub=N_SUB_BOOT)
logger.info(f"primary bootstrap done ({time.time() - t0:.0f}s, {N_BOOT} replicates per task)")

### 5. Sensitivity bootstraps

These are the replicates that the sensitivity versions of the rule need:
- s1: twin-segment hull;
- s4: template-cluster resampling;
- s5: slot-paired resampling;
- s3 / s8: Step-0 vertex alternatives;
- s6: the two other conventions for `other` programs.

In [ ]:
t0 = time.time()
e6 = {t: tds[t] for t in E6 if t in tds}
extra: dict[str, Any] = {}
Bs = N_BOOT_SENS
extra["s1"] = cached_bootstrap(e6, BOOT_CLASSES, Bs, BOOT_SEED + 11, edges="twin", workers=WORKERS, n_sub=N_SUB_BOOT)
extra["s4"] = cached_bootstrap(e6, BOOT_CLASSES, Bs, BOOT_SEED + 14, scheme="template_cluster",
                               workers=WORKERS, n_sub=N_SUB_BOOT)
extra["s5"] = cached_bootstrap(e6, BOOT_CLASSES, Bs, BOOT_SEED + 15, scheme="slot_paired", workers=WORKERS,
                               n_sub=N_SUB_BOOT)
extra["s3"] = boot_alt(e6, Bs, BOOT_SEED + 13, "s3", WORKERS)
extra["s8"] = boot_alt(e6, Bs, BOOT_SEED + 18, "s8", WORKERS)
logger.info(f"sensitivity bootstraps done ({time.time() - t0:.0f}s)")
if omap is not None:
    U = fixed_uniforms(N_SUB_POINT)
    for conv in ("exclude", "one"):
        tdc = {t: TaskData(t, task_rows(samples, reused, t), omap, conv) for t in E6 if t in tds}
        extra[f"s6_{conv}_point"] = {t: {c: (s["H5"] if (s := class_price(tdc[t], c, U)) else None)
                                         for c in BOOT_CLASSES} for t in tdc}
        extra[f"s6_{conv}"] = cached_bootstrap(tdc, BOOT_CLASSES, Bs, BOOT_SEED + 16, scheme=f"stratified_{conv}",
                                               workers=WORKERS, n_sub=N_SUB_BOOT)
logger.info(f"all sensitivity bootstraps done ({time.time() - t0:.0f}s)")

### 6. The C1 decision (primary rule and sensitivities)

In [ ]:
c1 = c1_all(point, boot, extra)
# bootstrap intervals of each class's H5 / P15 / P20 per task
cis = {}
for t, bt in boot.items():
    cis[t] = {}
    for c, d in bt.items():
        p15 = [x for x in d["P15"]]
        cis[t][c] = {"H5_ci95": pct_ci(d["H5"]), "P15_ci95": pct_ci(p15), "P20_ci95": pct_ci(d["P20"]),
                     "share_reaching_1.5": float(np.mean([x is not None for x in p15])) if p15 else None,
                     "share_reaching_2.0": float(np.mean([x is not None for x in d["P20"]])) if d["P20"] else None}
c1["class_ci"] = cis
logger.info(f"C1 primary: {c1['primary']['verdict']} (count1 {c1['primary']['count1']}, count2 "
            f"{c1['primary']['count2']}, D_bar {c1['primary']['D_bar']}, CI {c1['primary']['D_bar_ci95']}); "
            f"analysis {time.time() - t_analysis:.0f}s")

### 7. Descriptive analyses (`src/analysis/descriptive.py`, verbatim subset)

This cell runs three of the descriptive analyses:
- **(c) alpha sweep:** pass rate, off-default share and Hill at alpha = 0 (TWIN), 0.5, 1, 2, 4 and 8. The breakage
  point is the first alpha whose pass rate falls below 0.8 x TWIN's.
- **(e) CRN flips:** for each arm, the probability that the program differs from TWIN's at the *same* CRN slot (same
  template, same uniforms). It is given twice: for the code text (a new phrasing) and for the mechanism label (a new
  idea).
- **(g) per-template off-default shares.**

Four analyses are left out:
- (a) the non-movable tasks and (b) PLAIN vs TWIN need arms outside the demo subset;
- (d) ideas vs phrasings needs the full program texts;
- (f) the forecast score needs the committed screen forecast.

The realised PN displacements (`data/realised.jsonl`) are not included either, so the sweep omits its displacement
columns.

In [ ]:
ALPHA_ARMS = [("TWIN", 0.0), ("PN_a0.5", 0.5), ("PN_a1", 1.0), ("PN_a2", 2.0), ("PN_a4", 4.0), ("PN_a8", 8.0)]


def by_task_arm(rows: list[dict[str, Any]]) -> dict[str, dict[str, list[dict[str, Any]]]]:
    out: dict[str, dict[str, list[dict[str, Any]]]] = defaultdict(lambda: defaultdict(list))
    for r in rows:
        out[r["task"]][r["arm"]].append(r)
    return out


def correct(r: dict[str, Any]) -> bool:
    return bool(r.get("passed")) and r.get("label") != "invalid"


def offd(rs: list[dict[str, Any]], task: str) -> float:
    return sum(1 for r in rs if r["label"] not in (DEFAULT[task], "invalid")) / max(len(rs), 1)


def cell(rs: list[dict[str, Any]], task: str) -> dict[str, Any]:
    cc = Counter(r["label"] for r in rs if correct(r))
    return {"n": len(rs), "pass": float(np.mean([correct(r) for r in rs])) if rs else None,
            "invalid_share": sum(r["label"] == "invalid" for r in rs) / max(len(rs), 1),
            "cap_hit_share": sum(r.get("finish_reason") == "length" for r in rs) / max(len(rs), 1),
            "off_default_share": offd(rs, task),
            "off_default_share_invalid_counts": sum(1 for r in rs if r["label"] != DEFAULT[task]) / max(len(rs), 1),
            "correct_comp": dict(cc.most_common()),
            "attempted_comp": dict(Counter(r["label"] for r in rs).most_common()),
            "hill25_step0": step0_hill(cc) if sum(cc.values()) else None}


def alpha_sweep(T: dict, realised: list[dict[str, Any]]) -> dict[str, Any]:
    disp = defaultdict(list)
    for r in realised:
        disp[(r["task"], r["arm"])].append(r)
    out = {}
    for t in E6:
        if t not in T:
            continue
        tw = cell(T[t].get("TWIN", []), t)
        rows = []
        for arm, al in ALPHA_ARMS:
            rs = T[t].get(arm)
            if not rs:
                continue
            c = cell(rs, t)
            rec = disp.get((t, arm), [])
            if rec:
                c["disp_S_mean_b24_35"] = float(np.mean([np.mean([x["disp_S"][str(l)] if str(l) in x["disp_S"] else
                                                                  x["disp_S"][l] for l in range(24, 36)]) for x in rec]))
                c["disp_R_b35"] = float(np.mean([x["disp_R"].get("35", x["disp_R"].get(35)) for x in rec]))
                c["disp_S_b12_over_target"] = float(np.mean([x["disp_S"].get("12", x["disp_S"].get(12)) /
                                                             max(x["target_mean"].get("12", x["target_mean"].get(12)),
                                                                 1e-9) for x in rec]))
            c["alpha"] = al
            rows.append(c)
        brk = next((c["alpha"] for c in rows if c["pass"] is not None and tw["pass"] and
                    c["pass"] < 0.8 * tw["pass"]), None)
        out[t] = {"sweep": rows, "breakage_alpha": brk}
    return out


def flips(T: dict) -> dict[str, Any]:
    out = {}
    for t in sorted(T):
        tw = {(r["template"], r["slot"]): r for r in T[t].get("TWIN", [])}
        for arm, rs in sorted(T[t].items()):
            if arm == "TWIN" or arm.startswith(("VETO", "REUSED")):
                continue
            pairs = [(r, tw[(r["template"], r["slot"])]) for r in rs if (r["template"], r["slot"]) in tw]
            if not pairs:
                continue
            cd = [a.get("code_sha256") != b.get("code_sha256") for a, b in pairs]
            ld = [a["label"] != b["label"] for a, b in pairs]
            n_cd = sum(cd)
            out[f"{t}|{arm}"] = {"n": len(pairs), "P_code_differs": n_cd / len(pairs),
                                 "P_label_differs": sum(ld) / len(pairs),
                                 "P_label_differs_given_code_differs": (sum(1 for c, l in zip(cd, ld) if c and l) /
                                                                        n_cd) if n_cd else None}
    return out


def per_template(T: dict) -> dict[str, Any]:
    out = {}
    for t in E6:
        if t not in T:
            continue
        for arm, rs in sorted(T[t].items()):
            byw = defaultdict(list)
            for r in rs:
                byw[str(r["template"])].append(r)
            out[f"{t}|{arm}"] = {w: round(offd(v, t), 3) for w, v in sorted(byw.items())}
    return out


T = by_task_arm(samples)          # descriptive analyses use this run's rows only (data/samples.jsonl)
sweep = alpha_sweep(T, [])        # realised displacements (data/realised.jsonl) are not part of the demo data
fl = flips(T)
ptpl = per_template(T)
orig_fl = meta["original_results"]["flips"]
print("CRN flips identical to the full run:",
      sum(_same(v["P_label_differs"], orig_fl[k]["P_label_differs"]) and
          _same(v["P_code_differs"], orig_fl[k]["P_code_differs"]) for k, v in fl.items() if k in orig_fl), "/", len(fl))
print("breakage alpha per task:", {t: v["breakage_alpha"] for t, v in sweep.items()})
print("off-default share per template, T03 PN_a2:", ptpl["T03|PN_a2"])

## Results

### C1 per task, this run vs the full run

The H5 point estimates and D_t are identical to the paper once `N_SUB_POINT = 1000`. The bootstrap columns, i.e. the
95% CI of D_t and P(H5_PN > H5_best), approach the paper's values as `N_BOOT` grows.

In [ ]:
p, op = c1["primary"], meta["original_results"]["c1"]["primary"]
fmt = lambda v, d=2: "-" if v is None else f"{v:.{d}f}"
rows = []
for t in E6:
    d, o = p["per_task"][t], op["per_task"][t]
    rows.append({"task (default)": f"{t} ({DEFAULT[t]})", "H5 PN": fmt(d["H5_PN"]), "best NA knob": d["best_na"],
                 "H5 best NA": fmt(d["H5_best"]),
                 "D_t [95% CI]": f"{fmt(d['D_t'])} [{fmt(d['D_t_boot_ci95'][0])}, {fmt(d['D_t_boot_ci95'][1])}]",
                 "P(PN>best)": fmt(d["P_boot_PN_gt_best"]),
                 "full run: D_t [95% CI]": f"{fmt(o['D_t'])} [{fmt(o['D_t_boot_ci95'][0])}, {fmt(o['D_t_boot_ci95'][1])}]",
                 "full run: P": fmt(o["P_boot_PN_gt_best"]),
                 "P15 PN": fmt(point[t]["PN"]["P15"], 1), "P20 PN": fmt(point[t]["PN"]["P20"], 1)})
print(f"C1 (this run, N_BOOT={N_BOOT}): {p['verdict']}  failed {p['failed_thresholds']}  count1 {p['count1']}/6  "
      f"count2 {p['count2']}/6  D_bar {p['D_bar']:.3f} [{fmt(p['D_bar_ci95'][0], 3)}, {fmt(p['D_bar_ci95'][1], 3)}]")
print(f"C1 (full run, 2000 replicates): {op['verdict']}  failed {op['failed_thresholds']}  count1 {op['count1']}/6  "
      f"count2 {op['count2']}/6  D_bar {op['D_bar']:.3f} [{op['D_bar_ci95'][0]:.3f}, {op['D_bar_ci95'][1]:.3f}]")
pd.DataFrame(rows).set_index("task (default)")

### H5 of every knob class, and the sensitivities

The H5 table matches the paper's table "H5 per knob class on E6". MINP and VS are absent because their arms are not in
the demo subset. In the full run they reached H5 <= 1.17 and <= 1.29. Each sensitivity re-evaluates the whole rule.

In [ ]:
classes_shown = ["PN", "TEMP", "TOPK10", "NONCE", "NOVELTY", "SEMSEED", "VETO_1SHOT", "VETO_PERSIST", "VETO_ITER"]
h5_tab = pd.DataFrame({c: {t: point[t][c]["H5"] if c in point[t] else None for t in E6} for c in classes_shown})
display(h5_tab.round(2))

sens = []
oc1 = meta["original_results"]["c1"]
for k, v in c1.items():
    if not isinstance(v, dict) or "verdict" not in v:
        continue
    o = oc1.get(k, {})
    sens.append({"rule": k, "verdict": v["verdict"], "count1": v["count1"], "count2": v["count2"],
                 "D_bar [95% CI]": f"{v['D_bar']:.3f} [{fmt(v['D_bar_ci95'][0], 3)}, {fmt(v['D_bar_ci95'][1], 3)}]",
                 "full run verdict": o.get("verdict"),
                 "full run D_bar [95% CI]": f"{o['D_bar']:.3f} [{o['D_bar_ci95'][0]:.3f}, {o['D_bar_ci95'][1]:.3f}]" if o else "-"})
for k in ("s2_minp_vs_added", "s2b_minp_only_added"):
    o = oc1[k]
    sens.append({"rule": k + " (not in demo)", "verdict": "-", "count1": "-", "count2": "-", "D_bar [95% CI]": "-",
                 "full run verdict": o["verdict"],
                 "full run D_bar [95% CI]": f"{o['D_bar']:.3f} [{o['D_bar_ci95'][0]:.3f}, {o['D_bar_ci95'][1]:.3f}]"})
pd.DataFrame(sens).set_index("rule")

### Figure 1: price curves on the randomisation hull (`src/figures.py` `fig1`)

Each line is a knob class's **frontier**: the maximum correct-only Hill reachable at a pass cost <= c. Points mark the
single arms (hull vertices), with the 90% Step-0 intervals of the knob rows. The dotted vertical line is the 5-point
budget behind H5, and the dotted horizontal lines are the P15 / P20 thresholds.

**How to read it.**
- Temperature, top-k and the nonce ban stay flat at about 1.
- PN already reaches Hill of about 2.1 at zero pass cost on T03 and T10.
- On T06 only the reused iterated veto (VETO_ITER) moves.
- On T2 the semantic seed (SEMSEED) climbs to about 1.9 within the 5-point budget, while PN needs more than 20
  points to get there.

In [ ]:
# --- figure style of src/figures.py ---
plt.rcParams.update({"pdf.fonttype": 42, "ps.fonttype": 42, "font.size": 9, "axes.titlesize": 10,
                     "axes.spines.top": False, "axes.spines.right": False, "savefig.bbox": "tight"})
OKABE = ["#0072B2", "#E69F00", "#009E73", "#D55E00", "#CC79A7", "#56B4E9", "#F0E442", "#000000", "#999999",
         "#882255", "#44AA99", "#117733"]
CLASS_COL = {"PN": OKABE[3], "TEMP": OKABE[0], "TOPK10": OKABE[5], "NONCE": OKABE[1], "NOVELTY": OKABE[4],
             "SEMSEED": OKABE[2], "VETO_1SHOT": OKABE[8], "VETO_PERSIST": OKABE[9], "VETO_ITER": OKABE[10],
             "MINP": OKABE[7], "VS": OKABE[11]}


def save(fig, name: str) -> None:
    plt.show()  # notebook: show inline (the original wrote figures/<name>.pdf and .png)


def fig1(samples, reused, kr_rows) -> None:
    U = fixed_uniforms(300)
    kr = {(r["task"], r["arm"]): r for r in kr_rows}  # notebook: knob rows passed in (original: results/knob_rows.jsonl)
    fig, axes = plt.subplots(2, 3, figsize=(11, 6.5), sharey=True)
    cs = np.linspace(-5, 40, 91)
    for ax, t in zip(axes.ravel(), E6):
        td = TaskData(t, task_rows(samples, reused, t), omap)
        for cls, col in CLASS_COL.items():
            arms = class_vertices(td, cls)
            if arms is None:
                continue
            R = np.array([td.vertex(a)[0] for a in arms])
            P = np.array([td.vertex(a)[1] for a in arms])
            h = Hull(R, P, U)
            ys = [h.best_at_cost(c, refine=False)["hill"] for c in cs]
            ys = [np.nan if y is None else y for y in ys]
            ls = "--" if cls in ("MINP", "VS") else "-"
            ax.plot(cs, ys, color=col, lw=1.6 if cls == "PN" else 1.0, ls=ls, label=cls)
            for a, c in zip(arms[1:], h.cost_v[1:]):
                k = kr.get((t, a))
                if k and k.get("hill25_step0") is not None:
                    lo, hi = k["ci90"] if k["ci90"] and k["ci90"][0] is not None else (None, None)
                    ax.plot([c], [k["hill25_step0"]], "o", color=col, ms=3)
                    if lo is not None:
                        ax.plot([c, c], [lo, hi], color=col, lw=0.6)
        ax.axvline(5, color="0.5", lw=0.7, ls=":")
        for y in (1.5, 2.0):
            ax.axhline(y, color="0.7", lw=0.6, ls=":")
        ax.set_xlim(-5, 45)
        ax.set_title(f"{t} (default {DEFAULT[t]})")
    for ax in axes[1]:
        ax.set_xlabel("pass cost (points; vertices above 45 not shown)")
    axes[0, 0].set_ylabel("correct-only Hill-1 (depth 25)")
    axes[1, 0].set_ylabel("correct-only Hill-1 (depth 25)")
    hl = {}
    for ax in axes.ravel():
        for h_, l_ in zip(*ax.get_legend_handles_labels()):
            hl.setdefault(l_, h_)
    fig.legend(list(hl.values()), list(hl.keys()), loc="lower center", ncol=6, frameon=False,
               bbox_to_anchor=(0.5, -0.02))
    fig.suptitle("Fig. 1  Randomisation-hull frontier (max Hill at cost <= c) of every knob class; points: arm "
                 "vertices with 90% CIs (dashed: sensitivity knobs)")
    fig.tight_layout(rect=(0, 0.06, 1, 0.97))
    save(fig, "fig1_hill_vs_cost")


t0 = time.time()
fig1(samples, reused, kr)
print(f"({time.time() - t0:.1f}s)")

### Figures 2 and 3: what the knobs do to the programs

**Fig. 2** (the original `fig2`) shows the correct compositions of TWIN, PN alpha 1 and 2, T1.6 and NOVELTY, with the
failing and invalid shares.

**Fig. 3** (the original `fig3`) is the alpha dose-response. As alpha rises, the off-default share and Hill climb until
the pass rate breaks.

In [ ]:
def fig2(T) -> None:
    arms = ["TWIN", "PN_a1", "PN_a2", "T1.6", "NOVELTY"]
    fig, axes = plt.subplots(2, 3, figsize=(13, 7.5))
    for ax, t in zip(axes.ravel(), E6):
        labs = sorted({r["label"] for a in arms for r in T[t].get(a, []) if r.get("passed") and r["label"] != "invalid"})
        cols = {l: OKABE[i % len(OKABE)] for i, l in enumerate(labs)}
        for i, a in enumerate(arms):
            rs = T[t].get(a, [])
            if not rs:
                continue
            n = len(rs)
            left = 0.0
            cc = Counter(r["label"] for r in rs if r.get("passed") and r["label"] != "invalid")
            for l in labs:
                w = cc.get(l, 0) / n
                ax.barh(i, w, left=left, color=cols[l], edgecolor="white", lw=0.3,
                        label=l if i == 0 or l not in [x.get_label() for x in ax.patches] else None)
                left += w
            inv = sum(r["label"] == "invalid" for r in rs) / n
            fail = 1 - left - inv
            ax.barh(i, fail, left=left, color="0.85", edgecolor="white", lw=0.3)
            ax.barh(i, inv, left=left + fail, color="0.55", edgecolor="white", lw=0.3)
        ax.set_yticks(range(len(arms)))
        ax.set_yticklabels(arms)
        ax.set_xlim(0, 1)
        ax.set_title(t)
        handles = [plt.Rectangle((0, 0), 1, 1, color=cols[l]) for l in labs] + \
            [plt.Rectangle((0, 0), 1, 1, color="0.85"), plt.Rectangle((0, 0), 1, 1, color="0.55")]
        ax.legend(handles, labs + ["failing", "invalid"], fontsize=6.5, frameon=False, loc="upper center",
                  bbox_to_anchor=(0.5, -0.08), ncol=4)
    fig.suptitle("Fig. 2  Correct compositions (share of N = 50), failing and invalid shares")
    fig.tight_layout(rect=(0, 0, 1, 0.96), h_pad=3.0, w_pad=2.0)
    save(fig, "fig2_compositions")


def fig3(sw) -> None:
    # notebook: the sweep is passed in (original: read results/alpha_sweep.json)
    fig, axes = plt.subplots(1, 3, figsize=(11, 3.4))
    for i, t in enumerate(E6):
        if t not in sw:
            continue
        rows = sw[t]["sweep"]
        al = [r["alpha"] for r in rows]
        x = [a if a > 0 else 0.25 for a in al]
        axes[0].plot(x, [r["pass"] for r in rows], "o-", color=OKABE[i], label=t, ms=3)
        axes[1].plot(x, [r["off_default_share"] for r in rows], "o-", color=OKABE[i], ms=3)
        axes[2].plot(x, [r["hill25_step0"] for r in rows], "o-", color=OKABE[i], ms=3)
    for ax, yl in zip(axes, ("pass rate", "off-default share", "correct-only Hill-1")):
        ax.set_xscale("log", base=2)
        ax.set_xticks([0.25, 0.5, 1, 2, 4, 8])
        ax.set_xticklabels(["0", "0.5", "1", "2", "4", "8"])
        ax.set_xlabel("PN alpha (ban units)")
        ax.set_ylabel(yl)
    axes[0].legend(frameon=False, fontsize=7)
    fig.suptitle("Fig. 3  Dose-response of fresh per-sample plan noise (alpha 0 = TWIN)")
    fig.tight_layout(rect=(0, 0, 1, 0.93))
    save(fig, "fig3_alpha_dose_response")


fig2(T)
fig3(sweep)

### Ideas vs phrasings, and the forest plot of D_t

**Left: CRN flips.** At the same CRN slot as TWIN, how often does a knob change the *code* (x) and how often the
*algorithm* (y)?
- Points on the diagonal mean every code change is an algorithm change.
- Temperature sits far below the diagonal: it rewrites code but almost never the algorithm.
- PN at alpha >= 2 climbs toward it.

This scatter is a notebook addition, drawn from `flips.json`.

**Right: forest plot.** This is the original `fig6`: each task's D_t with its bootstrap CI, and the pooled D_bar.

In [ ]:
def fig6(c1_res) -> None:
    c1 = c1_res["primary"]  # notebook: the decision is passed in (original: read results/c1.json)
    per = c1["per_task"]
    ts = [t for t in E6 if t in per and per[t].get("D_t") is not None]
    fig, ax = plt.subplots(figsize=(6, 3.6))
    for i, t in enumerate(ts):
        d = per[t]
        lo, hi = d["D_t_boot_ci95"]
        ax.plot([lo, hi], [i, i], color=OKABE[0], lw=1.2)
        ax.plot([d["D_t"]], [i], "o", color=OKABE[0])
        ax.annotate(f"best NA: {d['best_na']}", (hi, i), fontsize=7, xytext=(4, -3), textcoords="offset points")
    lo, hi = c1["D_bar_ci95"]
    ax.plot([lo, hi], [len(ts), len(ts)], color=OKABE[3], lw=2)
    ax.plot([c1["D_bar"]], [len(ts)], "D", color=OKABE[3])
    ax.axvline(0, color="0.5", lw=0.7, ls=":")
    ax.set_yticks(range(len(ts) + 1))
    ax.set_yticklabels(ts + ["pooled D_bar"])
    ax.set_xlabel("D_t = H5(PN) - max over NA knobs of H5 (95% bootstrap CI)")
    ax.set_title(f"Fig. 6  C1: {c1['verdict']} (count1 {c1['count1']}, count2 {c1['count2']})")
    save(fig, "fig6_forest_Dt")


# notebook addition: CRN flips (descriptive (e)) -- code change vs algorithm change at the same CRN slot
fam = {"T1.3": "TEMP", "T1.6": "TEMP", "TOPK10": "TOPK10", "NONCE": "NONCE", "NOVELTY": "NOVELTY",
       "SEMSEED": "SEMSEED", "PN_a0.5": "PN", "PN_a1": "PN", "PN_a2": "PN", "PN_a4": "PN", "PN_a8": "PN"}
marks = dict(zip(E6, ["o", "s", "^", "D", "v", "P"]))
fig, ax = plt.subplots(figsize=(6.2, 4.6))
for k, v in fl.items():
    t, arm = k.split("|")
    if arm not in fam:
        continue
    col = CLASS_COL[fam[arm]]
    size = 14 + 10 * float(arm.split("_a")[1]) if arm.startswith("PN_a") else 22
    ax.scatter(v["P_code_differs"], v["P_label_differs"], color=col, marker=marks[t], s=size, alpha=0.8,
               edgecolor="white", lw=0.4)
ax.plot([0, 1], [0, 1], color="0.6", lw=0.7, ls=":")
ax.set_xlabel("P(code differs from TWIN at the same CRN slot)")
ax.set_ylabel("P(mechanism label differs)")
ax.set_xlim(-0.03, 1.03)
ax.set_ylim(-0.03, 1.03)
h1 = [plt.Line2D([], [], color=CLASS_COL[f], marker="o", ls="", label=f) for f in ("PN", "TEMP", "TOPK10", "NONCE",
                                                                                      "NOVELTY", "SEMSEED")]
h2 = [plt.Line2D([], [], color="0.4", marker=m, ls="", label=t) for t, m in marks.items()]
ax.legend(handles=h1 + h2, fontsize=7, frameon=False, ncol=2, loc="upper left")
ax.set_title("Ideas vs phrasings: CRN flips against TWIN (PN marker size ~ alpha)")
plt.show()

fig6(c1)
print("T1 (grid path, default BFS):", {a: (fl[f"T1|{a}"]["P_code_differs"], fl[f"T1|{a}"]["P_label_differs"])
                                        for a in ("T1.6", "PN_a2", "NOVELTY")}, "(P code differs, P label differs)")

### What the alternative programs look like

Below is one example program per correct mechanism, taken from a few arms. They show the default the model falls back
to and the alternatives that PN, the semantic seed and the veto unlock. Only the first 18 lines of each program are
shown.

In [ ]:
show = [("T03", ["TWIN", "PN_a2"]), ("T10", ["PN_a2"]), ("T06", ["VETO_ITER1"]), ("T2", ["SEMSEED"])]
for task, arms_ in show:
    for ex in cells:
        if ex["task"] != task or ex["arm"] not in arms_:
            continue
        print(f"=============== {task} ({meta['task_names'][task]}) | arm {ex['arm']} | correct composition {ex['output']}")
        for lab, src in ex["example_programs"].items():
            if ex["arm"] != "TWIN" and lab == DEFAULT[task]:
                continue  # skip the default mechanism outside TWIN
            print(f"--- {lab}: {meta['label_glossary'][task].get(lab, '')}")
            lines = src.rstrip().splitlines()
            print("\n".join(lines[:18]) + ("\n    ..." if len(lines) > 18 else ""))
        print()

## Summary

**Rule outcome.** On the six decisive tasks, the analysis code reproduces the paper's point estimates exactly, and the
pre-registered rule returns **PARTIAL**.

**Where PN wins and loses.**
- PN is the cheapest knob on T03 (k-th smallest) and T10 (connected components), where it reaches Hill 2 at zero pass
  cost.
- It shows small, uncertain edges on T05 and T1.
- It loses decisively on T06, where only the reused iterated veto moves `hash_set`, and on T2, where the semantic seed
  is cheaper.

**The other knobs.** Temperature, top-k and the nonce ban buy no correct algorithmic diversity: they change the
phrasing, not the idea.

**Reproducing the full run.** Run `method.py` stage by stage in the original experiment folder (`method.py`, `src/`,
`pyproject.toml`). It needs an FP8-capable GPU with at least 16 GB, plus `OPENROUTER_API_KEY` for the judges and
audits:

```bash
uv venv .venv --python=3.12 && uv pip install --python .venv/bin/python -r pyproject.toml
huggingface-cli download Qwen/Qwen3-8B --revision b968826d9c46dd6066d109eabc6255188de91218
.venv/bin/python method.py prompts
.venv/bin/python method.py inputs
.venv/bin/python method.py screen
.venv/bin/python method.py gpu --deadline-utc <ISO time to stop starting batches> &   # waits for the commit
.venv/bin/python method.py label_ec                     # after the engine-check rows exist
.venv/bin/python method.py label &                      # follows data/samples_raw.jsonl until the GPU finishes
.venv/bin/python method.py commit                       # rule + forecast + hashes; releases stage 100
# after the GPU stage and the labeller finish:
.venv/bin/python method.py audit
.venv/bin/python method.py analyze                      # the stage this notebook re-runs
.venv/bin/python method.py validity
.venv/bin/python method.py report
```